
# Accelerating Data Science with Databricks AutoML

##  Predicting patient readmission risk: Single click deployment with AutoML

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/hls/patient-readmission/patient-risk-ds-flow-2.png?raw=true" width="700px" style="float: right; margin-left: 10px;" />


In this notebook, we will explore how to use Databricks AutoML to generate the best notebooks to predict our patient readmission risk and deploy our model in production.

Databricks AutoML allows you to quickly generate baseline models and notebooks. 

ML experts can accelerate their workflow by fast-forwarding through the usual trial-and-error and focus on customizations using their domain knowledge, and citizen data scientists can quickly achieve usable results with a low-code approach.


<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F04-Data-Science-ML%2F04.2-AutoML-patient-admission-risk&demo_name=lakehouse-hls-readmission&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-hls-readmission%2F04-Data-Science-ML%2F04.2-AutoML-patient-admission-risk&version=1">

In [0]:
%pip install databricks-sdk==0.39.0 mlflow==2.19.0
dbutils.library.restartPython()

  Using cached mlflow-2.19.0-py3-none-any.whl.metadata (30 kB)
  Using cached mlflow_skinny-2.19.0-py3-none-any.whl.metadata (31 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/623.0 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 317.4/623.0 kB 9.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.0/623.0 kB 9.2 MB/s eta 0:00:00
Using cached mlflow-2.19.0-py3-none-any.whl (27.4 MB)
Using cached mlflow_skinny-2.19.0-py3-none-any.whl (5.9 MB)
Using cached docker-7.1.0-py3-none-any.whl (147 kB)
Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
Using cached graphql_relay-3.2.0-py3-none-any.whl (16 kB)
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.30.0
    Not uninstalling databricks-sdk at /databricks/python3/

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_hls_readmission`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


### Getting our training dataset 

Let's use the our training dataset determinining the readmission after 30 days for all our population. We will use that as our training label and what we want our model to predict.

In [0]:
training_dataset = spark.table('training_dataset')
training_dataset.display()

BIRTHDATE,DEATHDATE,PREFIX,SUFFIX,MAIDEN,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,MARITAL_D,MARITAL_M,MARITAL_S,MARITAL_W,RACE_asian,RACE_black,RACE_hawaiian,RACE_native,RACE_other,RACE_white,ETHNICITY_hispanic,ETHNICITY_nonhispanic,GENDER_F,GENDER_M,patient_id,START,STOP,30_DAY_READMISSION,ENCOUNTER_ID,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,enc_length,ENCOUNTERCLASS_ambulatory,ENCOUNTERCLASS_emergency,ENCOUNTERCLASS_hospice,ENCOUNTERCLASS_inpatient,ENCOUNTERCLASS_outpatient,ENCOUNTERCLASS_wellness,age_at_encounter
1960-10-12,2020-11-06,Mr.,null,null,Fall River,Massachusetts,Bristol County,25005.0,2747,41.750834597751016,-71.0452961077422,162261.25,895558.02,30440,0,0,1,0,0,0,0,0,0,1,0,1,0,1,5646f81d-85f2-c2d2-aa1d-3c30a0413148,2002-04-24T03:33:13Z,2002-04-24T04:21:41Z,0,0003e61d-9662-5111-f6ad-48646551f28c,85.55,802.11,0.0,2908,0,0,0,0,1,0,41.530458590006845
1949-11-14,null,Ms.,null,null,Bolton,Massachusetts,Worcester County,null,0,42.42686500765992,-71.61398510432554,259195.31,1819638.61,817966,0,0,1,0,0,1,0,0,0,0,0,1,1,0,ec3eb7ce-a707-49d3-c67b-758d354b1927,1949-12-19T02:05:57Z,1949-12-19T02:20:57Z,0,0005e160-77df-b17d-7d70-8d34abca9441,136.8,493.85,0.0,900,0,0,0,0,0,1,0.09582477754962354
1920-08-21,null,Mr.,null,null,Cochituate,Massachusetts,Middlesex County,null,0,42.28831634372727,-71.30992405399289,123160.4,1023238.6,858253,0,1,0,0,0,0,0,0,0,1,0,1,0,1,6d241af0-a1b5-6223-1843-b44e39b85659,2022-10-10T01:50:33Z,2022-10-10T02:05:33Z,1,0006bcbf-7d8c-2dcf-9c01-31cba06578bd,85.55,234.71,187.76,900,1,0,0,0,0,0,102.13552361396304
1960-03-31,null,Mr.,null,null,Gloucester,Massachusetts,Essex County,25009.0,1930,42.60004264570648,-70.66116465835674,27815.18,944263.9,19864,0,1,0,0,0,0,0,0,0,1,0,1,0,1,e1810300-6248-ae6c-d847-4d3a09fdfddb,2019-11-14T16:23:42Z,2019-11-14T17:16:28Z,1,0018bc8d-1712-9426-4839-95addc260c48,85.55,8251.87,8216.87,3166,0,0,0,0,1,0,59.62217659137577
1915-01-17,null,Mr.,null,null,Randolph,Massachusetts,Norfolk County,25021.0,2368,42.15366529705856,-71.04899600973981,221553.61,758519.43,45887,0,1,0,0,0,0,0,0,0,1,0,1,0,1,a8992104-8997-4c74-3480-3b66fd23df64,2017-08-27T14:50:54Z,2017-08-27T15:37:43Z,1,001b4d2a-a50b-c76d-d8cc-defa5a6b78e8,85.55,1179.74,943.78,2809,0,0,0,0,1,0,102.60917180013689
1960-10-12,2020-11-06,Mr.,null,null,Fall River,Massachusetts,Bristol County,25005.0,2747,41.750834597751016,-71.0452961077422,162261.25,895558.02,30440,0,0,1,0,0,0,0,0,0,1,0,1,0,1,5646f81d-85f2-c2d2-aa1d-3c30a0413148,1964-03-18T03:33:13Z,1964-03-18T03:48:13Z,0,0025e9ef-e36b-ebbe-4b19-72aa70f9a5f6,136.8,714.45,0.0,900,0,0,0,0,0,1,3.430527036276523
1959-07-01,null,Mr.,null,null,Newburyport,Massachusetts,Essex County,25009.0,1950,42.77880086615183,-70.86912980298715,95258.37,47651.28,126121,0,1,0,0,0,0,0,0,0,1,0,1,0,1,bb559f9e-c1ca-8dae-b3d5-12d95505fb98,1977-08-24T11:58:21Z,1977-08-24T12:42:44Z,0,002ab918-4e72-3216-cff6-893b003b978b,136.8,704.2,0.0,2663,0,0,0,0,0,1,18.1492128678987
1967-10-01,null,Ms.,null,null,Chelsea,Massachusetts,Suffolk County,25025.0,2151,42.4199237058549,-71.02552900372218,29100.6,1899452.29,18605,0,0,1,0,0,0,0,0,0,1,1,0,1,0,4c341ab3-7ecd-3ee6-ab0f-3fb78214d75f,2021-06-07T08:05:41Z,2021-06-07T09:17:24Z,0,002fce7f-eb35-996c-322c-7b587d8cfe2b,85.55,516.95,466.95,4303,1,0,0,0,0,0,53.68377823408624
1942-02-15,null,Mr.,null,null,Hopedale,Massachusetts,Worcester County,25027.0,1747,42.07755232349039,-71.58019896186094,111381.26,67109.96,830656,1,0,0,0,0,0,0,0,0,1,0,1,0,1,f86f92bf-9a5c-2c4e-906d-fe231585d560,2002-11-24T11:35:56Z,2002-11-24T12:23:38Z,0,0034de1c-5e1a-080a-81d1-599aa4443606,136.8,853.36,0.0,2862,0,0,0,0,0,1,60.772073921971256
1961-01-15,null,Mrs.,null,Nikolaus26,Pocasset,Massachusetts,Barnstable County,25001.0,2559,41.718558248476434,-70.57950414071792,595537.14,1067186.94,59555,1,0,0,0,0,0,0,0,0,1,0,1,1,0,377d69fa-d5e4-085a-cbda-6fabb1fb536e,1962-03-18T13:40:35Z,1962-03-18T13:55:35Z,0,00391881-6f97-9c35-d1d7-fe52b40e78f7,85.55,85.55,0.0,900,0,

### Define what features to look up for our model

Let's only keep the relevant features for our model training. We are removing columns such as `SSN` or `IDs`.

This step could also be done selecting the training_dataset table from the UI and selecting the column of interest.

*Note: this could also be retrived from our Feature Store tables. For more details on that open the companion notebook.*

In [0]:
feature_names = ['MARITAL_M', 'MARITAL_S', 'RACE_asian', 'RACE_black', 'RACE_hawaiian', 'RACE_other', 'RACE_white', 'ETHNICITY_hispanic', 'ETHNICITY_nonhispanic', 'GENDER_F', 'GENDER_M', 'INCOME'] \
              + ['BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'enc_length', 'ENCOUNTERCLASS_ambulatory', 'ENCOUNTERCLASS_emergency', 'ENCOUNTERCLASS_hospice', 'ENCOUNTERCLASS_inpatient', 'ENCOUNTERCLASS_outpatient', 'ENCOUNTERCLASS_wellness'] \
              + ['age_at_encounter'] \
              + ['30_DAY_READMISSION']


## Accelerating patient readmission model creation using MLFlow and Databricks AutoML
 
MLFlow is an open source project allowing model tracking, packaging and deployment. Every time your Data Science team works on a model, Databricks will track all parameters and data used and will auto-log them. This ensures ML traceability and reproductibility, making it easy to know what parameters/data were used to build each model and model version.

### A glass-box solution that empowers data teams without taking control away

While Databricks simplifies model deployment and governance (MLOps) with MLFlow, bootstraping new ML projects can still be a long and inefficient process.

Instead of creating the same boilerplate for each new project, Databricks AutoML can automatically generate state of the art models for Classifications, Regression, and Forecasting.


<img width="1000" src="https://github.com/QuentinAmbard/databricks-demo/raw/main/retail/resources/images/auto-ml-full.png"/>


Models can be directly deployed, or instead leverage generated notebooks to boostrap projects with best-practices, saving you weeks worth of effort.

<br style="clear: both">

<img style="float: right" width="600" src="https://raw.githubusercontent.com/borisbanushev/CAPM_Databricks/main/MLFlowAutoML.png"/>

### Using Databricks Auto ML with our readmission risk

AutoML is available in the "Machine Learning" menu. All we have to do is start a new AutoML Experiments and select the feature table we just created (`creditdecisioning_features`)

Our prediction target is the `30_DAY_READMISSION` column.

Click on Start, and Databricks will do the rest.

While this is done using the UI, you can also leverage the [python API](https://docs.databricks.com/applications/machine-learning/automl.html#automl-python-api-1)

In [0]:
import mlflow
model_name = "dbdemos_hls_patient_readmission"
xp_path = "/Shared/dbdemos/experiments/lakehouse-patient-admission"
xp_name = f"automl_churn_{datetime.now().strftime('%Y-%m-%d_%H:%M:%S')}"
try:
    from databricks import automl
    automl_run = automl.classify(
        experiment_name = xp_name,
        experiment_dir = xp_path,
        dataset = training_dataset.select(feature_names),
        target_col = "30_DAY_READMISSION",
        primary_metric="roc_auc",
        timeout_minutes = 10
    )
    #Make sure all users can access dbdemos shared experiment
    DBDemos.set_experiment_permission(f"{xp_path}/{xp_name}")
except Exception as e:
    if "cannot import name 'automl'" in str(e) or 'method_whitelist' in str(e):
        # Note: cannot import name 'automl' from 'databricks' likely means you're using serverless. Dbdemos doesn't support autoML serverless API - this will be improved soon.
        # Adding a temporary workaround to make sure it works well for now - ignore this for classic run
        automl_run = DBDemos.create_mockup_automl_run(f"{xp_path}/{xp_name}", training_dataset.select(feature_names).toPandas(), model_name = model_name, target_col = "30_DAY_READMISSION")
    else:
        raise e

(2025-11-03 22:29:30) WARNING: Hyperopt is deprecated for Databricks runtime for machine learning and will not be pre-installed in the next major version.
2025/11/03 22:29:37 INFO databricks.automl.client.manager: AutoML will optimize for ROC/AUC metric, which is tracked as val_roc_auc in the MLflow experiment.
2025/11/03 22:29:37 INFO databricks.automl.shared.databricks_utils: No host name to create absolute URL
2025/11/03 22:29:37 INFO databricks.automl.client.manager: MLflow Experiment ID: 2876013310725870
2025/11/03 22:29:37 INFO databricks.automl.client.manager: MLflow Experiment: #mlflow/experiments/2876013310725870


🏃 View run sneaky-robin-497 at: https://xxxx.cloud.databricks.com/ml/experiments/2876013310725870/runs/c74353f99f6d4b808b8abff2cc31304c
🧪 View experiment at: https://xxxx.cloud.databricks.com/ml/experiments/2876013310725870


2025/11/03 22:30:50 INFO databricks.automl.shared.databricks_utils: No host name to create absolute URL
2025/11/03 22:30:50 INFO databricks.automl.client.manager: Data exploration notebook: #notebook/2876013310725876
2025/11/03 22:40:26 INFO databricks.automl.client.manager: AutoML experiment completed successfully.


,Train,Validation,Test
roc_auc,0.966,0.950,0.949
recall_score,0.873,0.850,0.846
false_negatives,5463.000,2161.000,2201.000
false_positives,3339.000,1491.000,1447.000
example_count,85403.000,28543.000,28449.000
precision_score,0.918,0.891,0.893
true_positives,37536.000,12237.000,12124.000
precision_recall_auc,0.971,0.958,0.957
true_negatives,39065.000,12654.000,12677.000
log_loss,0.237,0.277,0.281


Experiment on /Shared/dbdemos/experiments/lakehouse-patient-admission/automl_churn_2025-11-03_22:29:28 was set public


## Deploying our model in production

Our model is now ready. We can review the notebook generated by the auto-ml run and customize if if required.

For this demo, we'll consider that our model is ready and deploy it in production in our Unity Catalog Model Registry:

In [0]:
#Enable Unity Catalog with mlflow registry
mlflow.set_registry_uri('databricks-uc')
    
model_registered = mlflow.register_model(f"runs:/{automl_run.best_trial.mlflow_run_id}/model", f"{catalog}.{db}.{model_name}")

#Move the model in production
print("registering model version "+model_registered.version+" as production model")
client = mlflow.tracking.MlflowClient()
client.set_registered_model_alias(name=f"{catalog}.{db}.{model_name}", alias="prod", version=model_registered.version)

#Make sure all other users can access the model for our demo
#DBDemos.set_model_permission(f"{catalog}.{db}.{model_name}", "ALL_PRIVILEGES", "account users")

Registered model 'main.dbdemos_hls_readmission.dbdemos_hls_patient_readmission' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

registering model version 12 as production model


Created version '12' of model 'main.dbdemos_hls_readmission.dbdemos_hls_patient_readmission'.


We just moved our automl model as production ready! 

Open the Unity Catalog [the dbdemos_hls_patient_readmission model](/explore/data/models/dbdemos/hls_patient_readmission/dbdemos_hls_patient_readmission) to explore its artifact and analyze the parameters used, including traceability to the notebook used for its creation.


## Our model predicting default risks is now deployed in production

So far we have:
* ingested all required data in a single source of truth using the OMOP data model,
* properly secured all data (including granting granular access controls, masked PII data, applied column level filtering),
* enhanced that data through feature engineering (and Feature Store as an option),
* used MLFlow AutoML to track experiments and build a machine learning model,
* registered the model.

### Next steps
We're now ready to use our model use it for:

- Batch inferences in notebook [04.3-Batch-Scoring-patient-readmission]($./04.3-Batch-Scoring-patient-readmission) to start using it for identifying patient at risk and providing cusom care to reduce readmission risk,
- Real time inference with [04.4-Model-Serving-patient-readmission]($./04.4-Model-Serving-patient-readmission) to enable realtime capabilities and instantly get insight for a specific patient.
- Explain model for our entire population or a specific patient to understand the risk factors and further personalize care with [04.5-Explainability-patient-readmission]($./04.5-Explainability-patient-readmission)